In [1]:
import pandas as pd
from sklearn.preprocessing import minmax_scale

In [3]:
df = pd.read_csv("../src/data/files/Sample_Data_F2.csv")
# df_mae_p = pd.read_csv("mae_producto_20rows.csv")
"""
Columns
= alternative
== same
1. Tipo subestrategia == DES_TIPO_SUBESTRATEGIA
2. Tipo grupo == DES_TIPO_GRUPO
3. Indicator padre == ES_PADRE
4. Indicator gratis = ES_GRATIS
5. Factor of repetition == FACTOR_REPETICION

"""

'\nColumns\n= alternative\n== same\n1. Tipo subestrategia == DES_TIPO_SUBESTRATEGIA\n2. Tipo grupo == DES_TIPO_GRUPO\n3. Indicator padre == ES_PADRE\n4. Indicator gratis = ES_GRATIS\n5. Factor of repetition == FACTOR_REPETICION\n\n'

In [4]:
print(df.shape)
df_filtered = df[df["CODCUC"] != "XXXXXXXXX"]
# print(df_filtered)
print(df_filtered.shape)

(1588, 28)
(865, 28)


In [5]:
df_filtered["Composite_key"] = df_filtered[["DES_TIPO_SUBESTRATEGIA","DES_TIPO_GRUPO","CODCUC","ES_PADRE","ES_GRATIS","FACTOR_REPETICION"]].astype(str).agg('|'.join, axis=1)

/var/folders/bh/74g24pss1j3_fnkxfr5pf0br0000gn/T/ipykernel_27830/3488556997.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtered["Composite_key"] = df_filtered[["DES_TIPO_SUBESTRATEGIA","DES_TIPO_GRUPO","CODCUC","ES_PADRE","ES_GRATIS","FACTOR_REPETICION"]].astype(str).agg('|'.join, axis=1)


In [6]:
df_filtered = df_filtered.drop("COMPOSITE_PRIMARY_KEY", axis=1)  # Remove "Column2"


In [8]:
def cal_recency(ref_date, last_purchase_date):
    last_purchase_date = pd.Timestamp(last_purchase_date)

    # Calculate the difference in days
    days_difference = (ref_date - last_purchase_date).days

    # Invert the difference for recency
    recency = 1 / (days_difference + 1)  
    return recency

# Reference date
reference_date = pd.Timestamp('2024-12-29') # We can also use system date by doing time.time()

# Calculate recency
recency_values = [cal_recency(reference_date, date) for date in df_filtered['FECHAPROCESO']]

df_filtered["recency"] = recency_values

df_filtered.to_csv("zero_phase.csv", index=False)

print(df_filtered)

     CODPAIS  ANIOCAMPANA  CODEBELISTA  CODVENTA  CODPRODUCTOSAP  ID_OFERTA  \
0         PE       202304     49946996     38332       200102550        539   
2         PE       202302     49946996    104020       200083615       3114   
4         PE       202304     10961556      5977       200106608        738   
5         PE       202309     50379884      1636       200106397        817   
6         PE       202305     43985620    121807       200090452       4569   
...      ...          ...          ...       ...             ...        ...   
1560      PE       202305     35545956     35599       200095170       1964   
1561      PE       202305     50073300     42262       200106443       2162   
1571      PE       202303     50020096    202646       200105188       3362   
1572      PE       202303     49389205     38114       200094946       1161   
1587      PE       202305     43985620     15125       200106290       1786   

      COD_CATALOGO  ID_MACROESTRATEGIA  ID_TIPO_SUB

In [9]:
concatenateds = df_filtered.groupby("ID_OFERTA")[["Composite_key", "CODEBELISTA", "recency"]].agg(
    {
        'CODEBELISTA': lambda x: x.dropna().tolist(),
        'Composite_key': lambda x: '|'.join(x),
        'recency': 'mean'  # Aggregating recency by mean
    }
).reset_index()
# Expand the column C into separate rows
expanded_df = concatenateds.explode('CODEBELISTA')

print(expanded_df)
# ave DataFrame to a CSV file
expanded_df.to_csv("second_phase.csv", index=False)
# print(concatenateds.shape)

     ID_OFERTA CODEBELISTA                                      Composite_key  \
0            2    35545956      INDIVIDUAL + ADICIONAL|FIJO|200101086|1|0|1.0   
1            5    10961556            INDIVIDUAL|VARIABLE|P0292127008|1|0|1.0   
2           23    50379884  VOLUMEN|VARIABLE|200039966|1|0|1.0|VOLUMEN|VAR...   
2           23    49946996  VOLUMEN|VARIABLE|200039966|1|0|1.0|VOLUMEN|VAR...   
2           23    45871533  VOLUMEN|VARIABLE|200039966|1|0|1.0|VOLUMEN|VAR...   
..         ...         ...                                                ...   
466       4569    43985620  993|VARIABLE|200040636|1|0|1.0|993|VARIABLE|20...   
466       4569    43985620  993|VARIABLE|200040636|1|0|1.0|993|VARIABLE|20...   
467       4661    45871533                     VOLUMEN|FIJO|200084783|1|0|2.0   
468       4802    50020096                     VOLUMEN|FIJO|200083596|1|0|1.0   
469       4836    50020096                     VOLUMEN|FIJO|200109875|1|0|1.0   

      recency  
0    0.0014

In [10]:
# Group by CODEBELISTA and Composite_key, aggregating ID_OFERTA into a list and calculating count
grouped_df = expanded_df.groupby(['CODEBELISTA', 'Composite_key']).agg({
    'ID_OFERTA': lambda x: x.tolist(),  # convert ID_OFERTA to list,
    'recency':'mean'
}).reset_index()

# Add the count column
grouped_df['count'] = expanded_df.groupby(['CODEBELISTA', 'Composite_key']).size().values

# Normalize the 'value' column to be between 0 and 1
grouped_df['normalized_count'] = minmax_scale(grouped_df['count'])
grouped_df['normalized_recency'] = minmax_scale(grouped_df['recency'])

print(grouped_df)

grouped_df.to_csv("third_phase.csv", index=False)

     CODEBELISTA                                      Composite_key  \
0       10961556  INDIVIDUAL + ADICIONAL|FIJO|200083596|1|0|1.0|...   
1       10961556  INDIVIDUAL + ADICIONAL|FIJO|200109874|1|0|1.0|...   
2       10961556                  INDIVIDUAL|FIJO|200043493|1|0|1.0   
3       10961556                  INDIVIDUAL|FIJO|200058353|1|0|1.0   
4       10961556  INDIVIDUAL|FIJO|200062879|1|0|1.0|INDIVIDUAL|F...   
..           ...                                                ...   
559     50379884                    SET FIJO|FIJO|200105208|1|0|1.0   
560     50379884  VOLUMEN|FIJO|P0210060001|0|0|1.0|VOLUMEN|FIJO|...   
561     50379884  VOLUMEN|VARIABLE|200038319|1|0|1.0|VOLUMEN|VAR...   
562     50379884  VOLUMEN|VARIABLE|200039966|1|0|1.0|VOLUMEN|VAR...   
563     50379884               VOLUMEN|VARIABLE|P0216050000|1|0|1.0   

        ID_OFERTA   recency  count  normalized_count  normalized_recency  
0           [172]  0.001469      1          0.000000            0.194156

In [11]:

#Score Calculation
def score_calculation(recency,frequency,weight1=0.3,weight2=0.7):
    score = weight1*recency + frequency*weight2
    return score

grouped_df['score'] = grouped_df.apply(lambda row: score_calculation(row['normalized_recency'], row['normalized_count']), axis=1)

# Sorting score in descending order
sorted_df = grouped_df.sort_values('score', ascending=False)

print(grouped_df)

sorted_df.to_csv("fourth_phase.csv", index=False)


     CODEBELISTA                                      Composite_key  \
0       10961556  INDIVIDUAL + ADICIONAL|FIJO|200083596|1|0|1.0|...   
1       10961556  INDIVIDUAL + ADICIONAL|FIJO|200109874|1|0|1.0|...   
2       10961556                  INDIVIDUAL|FIJO|200043493|1|0|1.0   
3       10961556                  INDIVIDUAL|FIJO|200058353|1|0|1.0   
4       10961556  INDIVIDUAL|FIJO|200062879|1|0|1.0|INDIVIDUAL|F...   
..           ...                                                ...   
559     50379884                    SET FIJO|FIJO|200105208|1|0|1.0   
560     50379884  VOLUMEN|FIJO|P0210060001|0|0|1.0|VOLUMEN|FIJO|...   
561     50379884  VOLUMEN|VARIABLE|200038319|1|0|1.0|VOLUMEN|VAR...   
562     50379884  VOLUMEN|VARIABLE|200039966|1|0|1.0|VOLUMEN|VAR...   
563     50379884               VOLUMEN|VARIABLE|P0216050000|1|0|1.0   

        ID_OFERTA   recency  count  normalized_count  normalized_recency  \
0           [172]  0.001469      1          0.000000            0.19415